In [16]:
# trial example 
ROUTE        = "29"
DIRECTION    = 0
SERVICE_DATE = "2026-06-18"
DATA_MODE    = "cache"

In [17]:
# setup
from scripts.analysis.offline_debug import setup, _load_static, _stage_label
from datetime import date

sd = date.fromisoformat(SERVICE_DATE)

ctx = setup(data_mode=DATA_MODE)
if not hasattr(ctx, "STATIC"):
    _load_static(ctx)

labels = _stage_label(sd, ROUTE, DIRECTION, ctx, view_examples_as_df=True)
print(f"Total labelled examples: {len(labels)}")

if labels.empty or labels.isna().all().all():
    raise RuntimeError("No examples found. Check route/date or prime cache with DATA_MODE='live'")

Total labelled examples: 1335


In [18]:
# AUDIT:

# explode headway_labels into one row per (example, horizon)
hw_exploded = labels["headway_labels"].explode()
sp_exploded = labels["labels"].explode()

# counts per slot
num_hw_valid   = hw_exploded.notna().sum()
num_hw_nan     = hw_exploded.isna().sum()
num_sp_valid   = sp_exploded.notna().sum()
total_slots    = len(hw_exploded)

print(f"Total label slots (examples x horizons): {total_slots}")
print()
print(f"Headway label (headway_labels_s):")
print(f"  Valid : {num_hw_valid} ({100*num_hw_valid/total_slots:.1f}%)")
print(f"  NaN   : {num_hw_nan}  ({100*num_hw_nan/total_slots:.1f}%)")
print()
print(f"Spatial label (labels):")
print(f"  Valid : {num_sp_valid} ({100*num_sp_valid/total_slots:.1f}%)")


Total label slots (examples x horizons): 40050

Headway label (headway_labels_s):
  Valid : 14775 (36.9%)
  NaN   : 25275  (63.1%)

Spatial label (labels):
  Valid : 14775 (36.9%)


In [30]:
# Flag if target routes are interlined/branching

import pandas as pd
trips = pd.read_csv("Complete GTFS/trips.txt")

# check if route has multiple shape_ids (branching indicator)
trips_29 = trips[trips["route_id"] == 29]
print("Distinct shape_ids for route 29:")
print(trips_29["shape_id"].value_counts())

# check for interlining — same block_id appears on multiple routes
blocks = trips.groupby("block_id")["route_id"].nunique()
interlined_blocks = blocks[blocks > 1]
route_29_blocks = trips_29["block_id"].unique()
interlined_29 = [b for b in route_29_blocks if b in interlined_blocks.index]
print(f"\nInterlined blocks on route 29: {len(interlined_29)}")

Distinct shape_ids for route 29:
shape_id
shp-29-07    494
shp-29-56    487
shp-29-17      5
shp-29-63      5
shp-29-19      3
shp-29-64      3
Name: count, dtype: int64

Interlined blocks on route 29: 40


/var/folders/b2/y50nhjkn7554xcj0cffkg3m80000gn/T/ipykernel_42416/677773513.py:4: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  trips = pd.read_csv("Complete GTFS/trips.txt")
